 1. IMPORT LIBRARIES

In [1]:
import os
import kagglehub
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


In [2]:
data_path = r"C:\Users\SREEKUTTY\.cache\kagglehub\datasets\selonamaris\large-industrial-pump-maintenance-dataset\versions\1\Large_Industrial_Pump_Maintenance_Dataset.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [3]:

print("Dataset loaded successfully.")
print("Shape:", df.shape)


Dataset loaded successfully.
Shape: (20000, 8)


3. DEFINE FEATURES AND TARGET

In [4]:
features = [
    "Temperature",
    "Vibration",
    "Pressure",
    "Flow_Rate",
    "RPM",
    "Operational_Hours"
]

target = "Maintenance_Flag"

X = df[features]
y = df[target]

print("Features:")
print(features)

print("\nTarget:")
print(target)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Features:
['Temperature', 'Vibration', 'Pressure', 'Flow_Rate', 'RPM', 'Operational_Hours']

Target:
Maintenance_Flag

X shape: (20000, 6)
y shape: (20000,)


4. TRAIN / TEST SPLIT

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTest class distribution:")
print(y_test.value_counts())


Training set: (16000, 6)
Test set: (4000, 6)

Training class distribution:
Maintenance_Flag
0    8026
1    7974
Name: count, dtype: int64

Test class distribution:
Maintenance_Flag
0    2006
1    1994
Name: count, dtype: int64


5. LOGISTIC REGRESSION

In [6]:

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42))
])

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

logistic_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, logistic_pred),
    "Precision": precision_score(y_test, logistic_pred, zero_division=0),
    "Recall": recall_score(y_test, logistic_pred, zero_division=0),
    "F1": f1_score(y_test, logistic_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, logistic_prob)
}

print("Logistic Regression Results")
print("--------------------------------")
for metric, value in logistic_results.items():
    if metric != "Model":
        print(f"{metric}: {value:.4f}")



Logistic Regression Results
--------------------------------
Accuracy: 0.4953
Precision: 0.4930
Recall: 0.4443
F1: 0.4674
ROC-AUC: 0.5016


 6. DECISION TREE — INITIAL MODEL

In [7]:
tree_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)
tree_prob = tree_model.predict_proba(X_test)[:, 1]

tree_results = {
    "Model": "Decision Tree (depth=5)",
    "Accuracy": accuracy_score(y_test, tree_pred),
    "Precision": precision_score(y_test, tree_pred, zero_division=0),
    "Recall": recall_score(y_test, tree_pred, zero_division=0),
    "F1": f1_score(y_test, tree_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, tree_prob)
}

print("\nDecision Tree Results")
print("--------------------------------")
for metric, value in tree_results.items():
    if metric != "Model":
        print(f"{metric}: {value:.4f}")



Decision Tree Results
--------------------------------
Accuracy: 0.4960
Precision: 0.3750
Recall: 0.0165
F1: 0.0317
ROC-AUC: 0.4895


DECISION TREE HYPERPARAMETER TUNING
  5-FOLD CROSS-VALIDATION ON TRAINING DATA ONLY

In [8]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

tree_cv_results = []

for depth in depths:

    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )

    tree_cv_results.append({
        "max_depth": depth,
        "mean_roc_auc": scores.mean(),
        "std_roc_auc": scores.std()
    })

tree_cv_df = pd.DataFrame(tree_cv_results)

print("\nDecision Tree Cross-Validation Results")
display(tree_cv_df)




Decision Tree Cross-Validation Results


,max_depth,mean_roc_auc,std_roc_auc
0,1,0.499355,0.001834
1,2,0.496803,0.003112
2,3,0.496401,0.006817
3,4,0.499929,0.005613
4,5,0.498725,0.006934
5,6,0.498123,0.007990
6,8,0.497285,0.014660
7,10,0.496588,0.012645
8,15,0.502381,0.007868
9,20,0.506553,0.008780


SELECT BEST TREE DEPTH

In [9]:
best_depth = tree_cv_df.loc[
    tree_cv_df["mean_roc_auc"].idxmax(),
    "max_depth"
]

best_depth = int(best_depth)

print("\nSelected Decision Tree depth:", best_depth)



Selected Decision Tree depth: 20


 FINAL DECISION TREE

In [10]:

final_tree = DecisionTreeClassifier(
    max_depth=best_depth,
    random_state=42
)

final_tree.fit(X_train, y_train)

final_tree_pred = final_tree.predict(X_test)
final_tree_prob = final_tree.predict_proba(X_test)[:, 1]

final_tree_results = {
    "Model": f"Decision Tree (depth={best_depth})",
    "Accuracy": accuracy_score(y_test, final_tree_pred),
    "Precision": precision_score(
        y_test,
        final_tree_pred,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        final_tree_pred,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        final_tree_pred,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test,
        final_tree_prob
    )
}

print("\nFinal Decision Tree Results")
print("--------------------------------")

for metric, value in final_tree_results.items():
    if metric != "Model":
        print(f"{metric}: {value:.4f}")


Final Decision Tree Results
--------------------------------
Accuracy: 0.4998
Precision: 0.4984
Recall: 0.5542
F1: 0.5248
ROC-AUC: 0.5130


KNN HYPERPARAMETER TUNING

In [11]:
k_values = [3, 5, 7, 9, 11, 15, 21, 31]

knn_cv_results = []

for k in k_values:

    knn_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=k))
    ])

    scores = cross_val_score(
        knn_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )

    knn_cv_results.append({
        "k": k,
        "mean_roc_auc": scores.mean(),
        "std_roc_auc": scores.std()
    })

knn_cv_df = pd.DataFrame(knn_cv_results)

print("\nKNN Cross-Validation Results")
display(knn_cv_df)



KNN Cross-Validation Results


,k,mean_roc_auc,std_roc_auc
0,3,0.491702,0.008487
1,5,0.496401,0.007242
2,7,0.498929,0.010159
3,9,0.499405,0.011005
4,11,0.499548,0.010414
5,15,0.494927,0.012198
6,21,0.492503,0.012673
7,31,0.490477,0.014755


SELECT BEST K

In [12]:
best_k = knn_cv_df.loc[
    knn_cv_df["mean_roc_auc"].idxmax(),
    "k"
]

best_k = int(best_k)

print("\nSelected K:", best_k)


Selected K: 11


FINAL KNN

In [13]:
knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=best_k))
])

knn_model.fit(X_train, y_train)

knn_pred = knn_model.predict(X_test)
knn_prob = knn_model.predict_proba(X_test)[:, 1]

knn_results = {
    "Model": f"KNN (K={best_k})",
    "Accuracy": accuracy_score(y_test, knn_pred),
    "Precision": precision_score(
        y_test,
        knn_pred,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        knn_pred,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        knn_pred,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test,
        knn_prob
    )
}

print("\nFinal KNN Results")
print("--------------------------------")

for metric, value in knn_results.items():
    if metric != "Model":
        print(f"{metric}: {value:.4f}")


Final KNN Results
--------------------------------
Accuracy: 0.4888
Precision: 0.4872
Recall: 0.4850
F1: 0.4861
ROC-AUC: 0.4795


 MODEL COMPARISON

In [15]:
comparison_df = pd.DataFrame([
    logistic_results,
    final_tree_results,
    knn_results
])

print("\nModel Comparison")
display(comparison_df)


Model Comparison


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.49525,0.493044,0.444333,0.467423,0.501576
1,Decision Tree (depth=20),0.49975,0.498421,0.554162,0.524816,0.513047
2,KNN (K=11),0.48875,0.487154,0.484955,0.486052,0.479500


DECISION TREE FEATURE IMPORTANCE


In [16]:
feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": final_tree.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nDecision Tree Feature Importance")
display(feature_importance)


Decision Tree Feature Importance


,Feature,Importance
4,RPM,0.183252
2,Pressure,0.172413
0,Temperature,0.169884
1,Vibration,0.165389
5,Operational_Hours,0.164049
3,Flow_Rate,0.145013


In [17]:
import os
import joblib

# Create models folder
os.makedirs("models", exist_ok=True)

# Save the final Decision Tree
joblib.dump(
    final_tree,
    "models/decision_tree_model.pkl"
)

# Save the feature names
joblib.dump(
    features,
    "models/feature_names.pkl"
)

print("Model saved successfully.")
print("Feature names saved successfully.")

Model saved successfully.
Feature names saved successfully.


In [18]:
import os

print(
    "Model exists:",
    os.path.exists("models/decision_tree_model.pkl")
)

print(
    "Features file exists:",
    os.path.exists("models/feature_names.pkl")
)

Model exists: False
Features file exists: False


In [19]:
final_tree = DecisionTreeClassifier(
    max_depth=20,
    random_state=42
)

final_tree.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [20]:
import os
import joblib

# Create models folder
os.makedirs("models", exist_ok=True)

# Save final trained model
joblib.dump(
    final_tree,
    "models/decision_tree_model.pkl"
)

# Save feature names
joblib.dump(
    features,
    "models/feature_names.pkl"
)

print("Model saved successfully.")
print("Feature names saved successfully.")

Model saved successfully.
Feature names saved successfully.


In [21]:
import os

print("Model exists:", os.path.exists("models/decision_tree_model.pkl"))
print("Features exist:", os.path.exists("models/feature_names.pkl"))

Model exists: True
Features exist: True


In [22]:
import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(final_tree, "models/decision_tree_model.pkl")
joblib.dump(features, "models/feature_names.pkl")

print("Model saved successfully.")
print("Feature names saved successfully.")

Model saved successfully.
Feature names saved successfully.


In [23]:
import os

print("Model exists:", os.path.exists("models/decision_tree_model.pkl"))
print("Features exist:", os.path.exists("models/feature_names.pkl"))

Model exists: True
Features exist: True


In [24]:
import os
import joblib

# Exact project location
project_folder = r"D:\learn_depth ml\Industrial_Machine_Failure_Prediction"

# Create models folder
models_folder = os.path.join(project_folder, "models")
os.makedirs(models_folder, exist_ok=True)

# Exact model path
model_path = os.path.join(
    models_folder,
    "decision_tree_model.pkl"
)

# Save model
joblib.dump(final_tree, model_path)

print("Model saved to:")
print(model_path)

print("\nDoes the file exist?")
print(os.path.exists(model_path))

Model saved to:
D:\learn_depth ml\Industrial_Machine_Failure_Prediction\models\decision_tree_model.pkl

Does the file exist?
True


In [25]:
feature_path = os.path.join(
    models_folder,
    "feature_names.pkl"
)

joblib.dump(features, feature_path)

print("Features saved to:")
print(feature_path)

print("Does the file exist?")
print(os.path.exists(feature_path))

Features saved to:
D:\learn_depth ml\Industrial_Machine_Failure_Prediction\models\feature_names.pkl
Does the file exist?
True
